# 📚 Webアプリで利用するデータ(.csv)を作ります

以下の2つのファイルをご用意ください。

1. **配布CSV**（例：`[学科名ローマ字]_gakka.csv`）：ダウンロードしたフォルダ内の `Gakka/` フォルダに入っています。所属学科のものを選んでください。
2. **成績PDF**：3Sから「成績」タブ → 「成績照会」→ 右上の「PDF」でダウンロードされます。→ [3S](https://3s.musashi.ac.jp/uprx/up/pk/pky001/Pky00101.xhtml)

**使い方（順番に実行してください）**

1. ▶ **ステップ1** を実行 → ライブラリをインストール
2. ▶ **ステップ2** を実行 → 配布CSVをアップロード → 科目データを読み込み
3. ▶ **ステップ3** を実行 → 成績PDFをアップロード → 履修済み・単位状況を抽出
4. ▶ **ステップ4** を実行 → `merged_courses.csv` を生成してダウンロード

> **実行方法：** 各セルの左側にある ▶ ボタンをクリックするか、セルを選択して `Shift + Enter` を押してください。

In [ ]:
#@title ▶ ステップ1：準備（最初に1回だけ実行）

print("⏳ ライブラリをインストール中...")
!pip install pdfplumber -q
print("✅ 準備完了！ステップ2に進んでください。")

In [ ]:
#@title ▶ ステップ2：配布CSV（科目一覧）をアップロード → 科目データを読み込み

import pandas as pd
import io
from google.colab import files

# ── アップロード ───────────────────────────────────────────

print("📂 配布CSVファイル（例：mediashakai_courses.csv）を選択してください")
print()
uploaded = files.upload()

if not uploaded:
    print("❌ ファイルが選択されませんでした。もう一度実行してください。")
else:
    csv_filename = list(uploaded.keys())[0]
    csv_bytes    = list(uploaded.values())[0]
    print(f"\n✅ アップロード完了：{csv_filename}")
    print("\n⏳ 科目データを読み込み中...")

    # ── 読み込み ───────────────────────────────────────────

    df_courses = pd.read_csv(io.BytesIO(csv_bytes), encoding='utf-8-sig')

    # 履修済み・危険列はStep4で付け直すため除去
    df_courses = df_courses.drop(columns=[c for c in ['履修済み', '危険'] if c in df_courses.columns])

    course_rows = df_courses.to_dict('records')

    # ── 結果表示 ────────────────────────────────────────────

    summary = df_courses.groupby(
        [c for c in ['大分類', '中分類', '小分類', '必修選択'] if c in df_courses.columns]
    ).size().reset_index(name='件数')

    print(f"\n✅ 読み込み完了：{len(course_rows)} 件")
    print()
    print("【分類サマリー】")
    print(summary.to_string(index=False))
    print("\nステップ3に進んでください。")

In [ ]:
#@title ▶ ステップ3：成績PDFをアップロード → 履修済み・単位状況を抽出

import pdfplumber
from google.colab import files

# ── アップロード ───────────────────────────────────────────

print("📂 成績一覧表のPDFファイルを選択してください")
print()
uploaded = files.upload()

if not uploaded:
    print("❌ 成績PDFのアップロードがキャンセルされました。もう一度実行してください。")
    raise SystemExit

grades_pdf = list(uploaded.keys())[0]
print(f"\n✅ アップロード完了：{grades_pdf}")
print("\n⏳ 成績データを抽出中...")

# ── ヘルパー ───────────────────────────────────────────

def norm_width(s):
    """全角英数字・スラッシュを半角に正規化"""
    if not s:
        return ""
    result = []
    for c in s:
        code = ord(c)
        if 0xFF21 <= code <= 0xFF3A:
            result.append(chr(code - 0xFF21 + ord("A")))
        elif 0xFF41 <= code <= 0xFF5A:
            result.append(chr(code - 0xFF41 + ord("a")))
        elif 0xFF10 <= code <= 0xFF19:
            result.append(chr(code - 0xFF10 + ord("0")))
        elif c == "／":
            result.append("/")
        else:
            result.append(c)
    return "".join(result).strip()

def to_int(s):
    try:
        return int(norm_width(s or "").strip())
    except:
        return ""

def is_course_row(cells):
    if len(cells) < 5:
        return False
    return norm_width(cells[3]).isdigit() and len(norm_width(cells[3])) == 4 \
           and "学期" in norm_width(cells[4])

FAILED_GRADES = {"D", "/"}

# ── 履修済み科目の抽出（p.1〜3）────────────────────────

completed_names = set()
danger_names    = set()
grade_list      = []

with pdfplumber.open(grades_pdf) as pdf:
    for page in pdf.pages[:3]:
        tables = page.extract_tables()
        if not tables:
            continue
        for table in tables:
            for row in table:
                if not row:
                    continue
                cells = [norm_width(c) for c in row]
                if not is_course_row(cells):
                    continue
                name  = cells[0]
                grade = cells[2]
                grade_list.append({"科目名": name, "評価": grade})
                key = name.strip()
                if grade in FAILED_GRADES:
                    danger_names.add(key)
                else:
                    completed_names.add(key)

taken  = [g for g in grade_list if g["評価"] not in FAILED_GRADES]
failed = [g for g in grade_list if g["評価"] in FAILED_GRADES]

print(f"   単位取得済み：{len(taken)} 件")
if failed:
    print(f"   未取得（D・/）：{len(failed)} 件")
    for g in failed:
        print(f"     ⚠️  {g['科目名']}（評価: {g['評価']}）")

# ── 単位修得状況の抽出（単位修得状況表ページ）──────────

print("\n⏳ 単位修得状況を抽出中...")

credits_data = []

with pdfplumber.open(grades_pdf) as pdf:
    target = next(
        (pg for pg in pdf.pages if "単位修得状況" in (pg.extract_text() or "")),
        None
    )

if target is None:
    print("   ⚠️  単位修得状況表が見つかりませんでした（スキップ）")
else:
    parent_level = 0
    seen_total   = False

    for table in target.extract_tables():
        for row in table:
            if not row:
                continue
            cells = [norm_width(c or "").strip() for c in row]
            name  = cells[0] if cells else ""
            if not name or name == "科目分類" or "履修合計" in name:
                continue

            if name == "合計":
                if seen_total:
                    continue
                seen_total   = True
                level        = 0
                parent_level = 0
            elif name.startswith("◆"):
                level        = 1
                name         = name[1:].strip()
                parent_level = 1
            elif name.startswith("◇"):
                level        = 2
                name         = name[1:].strip()
                parent_level = 2
            else:
                level = 2 if parent_level <= 1 else 3

            credits_data.append({
                "name":     name,
                "level":    level,
                "required": to_int(cells[1]) if len(cells) > 1 else "",
                "earned":   to_int(cells[2]) if len(cells) > 2 else "",
                "enrolled": to_int(cells[3]) if len(cells) > 3 else "",
                "total":    to_int(cells[4]) if len(cells) > 4 else "",
            })

    INDENT = ["", "  ", "    ", "      "]
    for r in credits_data:
        pad  = INDENT[min(r["level"], 3)]
        req  = str(r["required"]) if r["required"] != "" else "─"
        earn = str(r["earned"])   if r["earned"]   != "" else "─"
        print(f"   {pad}{r['name']:<18}  要件 {req:>3}  取得 {earn:>3}")

print("\n✅ 抽出完了！ステップ4に進んでください。")

In [ ]:
#@title ▶ ステップ4：merged_courses.csv を生成してダウンロード

import csv
import io
from google.colab import files

# ── 事前チェック ───────────────────────────────────────────

errors = []
if "course_rows" not in dir() or not course_rows:
    errors.append("ステップ2（配布CSV）が完了していません")
if "completed_names" not in dir():
    errors.append("ステップ3（成績PDF）が完了していません")

if errors:
    for e in errors:
        print(f"❌ {e}")
    raise SystemExit

print("⏳ マージ中...")

# ── ヘルパー ───────────────────────────────────────────────

STRIP_SUFFIXES = [
    "（総合）", "（会話）", "（文系）", "（理系）",
    "［春学期］", "［秋学期］", "［春秋学期］", "［集中講義］",
]

def normalize_name(name):
    if not name:
        return ""
    s = name.replace("\u3000", " ")
    for suffix in STRIP_SUFFIXES:
        s = s.replace(suffix, "")
    return s.strip()

# ── フラグ付与 ─────────────────────────────────────────────

output_rows    = []
matched        = []
matched_danger = []
unmatched      = list(completed_names | danger_names)
fieldnames     = list(course_rows[0].keys()) + ["履修済み", "危険"]

for row in course_rows:
    key          = normalize_name(row.get("科目名", ""))
    is_completed = key in completed_names
    is_danger    = key in danger_names
    row["履修済み"] = "True" if is_completed else "False"
    row["危険"]     = "True" if is_danger    else "False"
    if is_completed:
        matched.append(row["科目名"])
        if key in unmatched: unmatched.remove(key)
    if is_danger:
        matched_danger.append(row["科目名"])
        if key in unmatched: unmatched.remove(key)
    output_rows.append(row)

# ── CSV出力（メモリ上で組み立て）──────────────────────────

buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=fieldnames)
writer.writeheader()
writer.writerows(output_rows)

if credits_data:
    buf.write("\n##CREDITS##\n")
    cred_writer = csv.DictWriter(buf, fieldnames=["name", "level", "required", "earned", "enrolled", "total"])
    cred_writer.writeheader()
    cred_writer.writerows(credits_data)

csv_text = buf.getvalue()
output_filename = "merged_courses.csv"

with open(output_filename, "w", encoding="utf-8-sig", newline="") as f:
    f.write(csv_text)

# ── 結果サマリー ──────────────────────────────────────────

print()
print("=" * 40)
print("✅ マージ完了")
print("=" * 40)
print(f"  科目一覧の総科目数  ：{len(output_rows)} 科目")
print(f"  履修済みマッチ      ：{len(matched)} 件")
print(f"  危険マッチ          ：{len(matched_danger)} 件")
print(f"  単位修得状況        ：{'✅ 追記済み' if credits_data else '─ なし'}")

if matched:
    print()
    print("【履修済み】")
    for name in matched:
        print(f"  ✅ {name}")

if matched_danger:
    print()
    print("【未取得（D・/）】")
    for name in matched_danger:
        print(f"  ⚠️  {name}")

if unmatched:
    print()
    print("【成績にあるが科目一覧で見つからなかった科目】")
    for name in sorted(unmatched):
        print(f"  ❓ {name}")

# ── ダウンロード ───────────────────────────────────────────

print()
print(f"⬇️  ダウンロード中: {output_filename}")
files.download(output_filename)
print("✅ ダウンロード完了！")
print()
print("📌 次のステップ")
print("   武蔵履修プランナー（rishu_planner.html）を開き、")
print(f"   {output_filename} を読み込んでください。")